# Reading sampled cells and downloading the related H5AD data from Census database 

In [1]:
from pathlib import Path
import os
import shutil

import numpy as np
import pandas as pd
import anndata as ad
import cellxgene_census
import scanpy as sc
from celltypist import models
from scipy.sparse import csr_matrix

import warnings
from anndata import ImplicitModificationWarning

CENSUS_VERSION = "2025-11-08"
MODEL_NAME = "Immune_All_Low.pkl"
CELL_KEY = "soma_joinid"
CHUNK_SIZE = 2_000
RESET_PARTS = False

/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [2]:
DATA_ROOT = Path(
    "/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE"
)
SAMPLE_ID_PATH = DATA_ROOT / "data" / "iha_sample_ids.csv"
BASE_DIR = DATA_ROOT / "iha_celltypist"
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Full-gene Census extract used as an intermediate and download checkpoint.
CENSUS_RAW_H5AD = BASE_DIR / "iha_expression_raw.h5ad"
PARTS_DIR = BASE_DIR / "_iha_extract_parts"

# Final files for the separate model-gene workflow.
MODEL_GENES_CSV = BASE_DIR / "celltypist_model_genes.csv"
MODEL_GENE_AUDIT_CSV = BASE_DIR / "celltypist_model_gene_match_audit.csv"
MODEL_RAW_H5AD = BASE_DIR / "iha_expression_model_genes_raw.h5ad"
MODEL_PREPARED_H5AD = BASE_DIR / "iha_expression_model_genes_prepared.h5ad"

print(f"Sample IDs: {SAMPLE_ID_PATH}")
print(f"Output folder: {BASE_DIR}")


Sample IDs: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/data/iha_sample_ids.csv
Output folder: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist


In [3]:
sample_ids = pd.read_csv(SAMPLE_ID_PATH)

if CELL_KEY not in sample_ids.columns:
    raise KeyError(f"{CELL_KEY!r} is missing from the sample ID file.")
if sample_ids[CELL_KEY].isna().any():
    raise ValueError("The sample ID file contains missing soma_joinid values.")
if sample_ids[CELL_KEY].duplicated().any():
    raise ValueError("The sample ID file contains duplicate soma_joinid values.")

expected_ids = np.sort(sample_ids[CELL_KEY].to_numpy(dtype=np.int64))

sample_meta = sample_ids.copy()
sample_meta[CELL_KEY] = sample_meta[CELL_KEY].astype("int64")
sample_meta = sample_meta.set_index(CELL_KEY)

# Download only this model if it is not already available in the CellTypist cache.
models.download_models(model=MODEL_NAME)
model = models.Model.load(MODEL_NAME)

model_genes = pd.Index(
    pd.Series(model.features).astype(str).unique(),
    name="model_gene",
)

pd.DataFrame({"model_gene": model_genes}).to_csv(MODEL_GENES_CSV, index=False)

print(f"Selected cells: {len(expected_ids):,}")
print(f"Genes in CellTypist model: {len(model_genes):,}")
print(f"Saved model gene list: {MODEL_GENES_CSV}")


📂 Storing models in /Users/artsvetkova/.celltypist/data/models
💾 Total models to download: 1
⏩ Skipping [1/1]: Immune_All_Low.pkl (file exists)


Selected cells: 54,225
Genes in CellTypist model: 6,639
Saved model gene list: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/celltypist_model_genes.csv


In [4]:
if RESET_PARTS and PARTS_DIR.exists() and not CENSUS_RAW_H5AD.exists():
    shutil.rmtree(PARTS_DIR)

PARTS_DIR.mkdir(parents=True, exist_ok=True)
chunk_starts = list(range(0, len(expected_ids), CHUNK_SIZE))
part_paths = [
    PARTS_DIR / f"part_{start:05d}.h5ad"
    for start in chunk_starts
]

if CENSUS_RAW_H5AD.exists():
    print(f"Using existing full-gene raw extract: {CENSUS_RAW_H5AD}")
else:
    def existing_part_matches(part_path, expected_chunk_ids):
        if not part_path.exists():
            return False

        fragment = None
        try:
            fragment = ad.read_h5ad(part_path, backed="r")
            if CELL_KEY not in fragment.obs.columns:
                return False

            fragment_ids = pd.to_numeric(
                fragment.obs[CELL_KEY], errors="raise"
            ).to_numpy(dtype=np.int64)

            return (
                len(fragment_ids) == len(expected_chunk_ids)
                and np.array_equal(np.sort(fragment_ids), expected_chunk_ids)
            )
        except Exception:
            return False
        finally:
            if fragment is not None and fragment.isbacked:
                fragment.file.close()

    with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
        for start, part_path in zip(chunk_starts, part_paths):
            coords = expected_ids[start : start + CHUNK_SIZE]

            if existing_part_matches(part_path, coords):
                print(f"Skipping verified part: {part_path.name}")
                continue

            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    message="Transforming to str index",
                    category=ImplicitModificationWarning,
                )
                adata_part = cellxgene_census.get_anndata(
                    census,
                    organism="Homo sapiens",
                    obs_coords=coords,
                    X_name="raw",
                )

            if CELL_KEY not in adata_part.obs.columns:
                raise KeyError(f"Census does not contain obs[{CELL_KEY!r}].")

            adata_part.obs_names = pd.Index(
                adata_part.obs[CELL_KEY].astype(str),
                name="cell_id",
            )

            got = pd.to_numeric(
                adata_part.obs[CELL_KEY], errors="raise"
            ).to_numpy(dtype=np.int64)

            if len(got) != len(coords) or not np.array_equal(np.sort(got), coords):
                missing = np.setdiff1d(coords, got)
                extra = np.setdiff1d(got, coords)
                raise RuntimeError(
                    f"Cell ID mismatch in chunk {start // CHUNK_SIZE}. "
                    f"Missing: {missing[:10].tolist()}; "
                    f"extra: {extra[:10].tolist()}"
                )

            meta = sample_meta.reindex(pd.Index(got, name=CELL_KEY))
            meta = meta.reset_index(drop=True)

            for column in meta.columns:
                if column in adata_part.obs.columns:
                    census_values = adata_part.obs[column].reset_index(drop=True)
                    if not census_values.astype("string").equals(
                        meta[column].astype("string")
                    ):
                        raise ValueError(
                            f"Sample CSV metadata column {column!r} "
                            "does not match Census."
                        )
                else:
                    adata_part.obs[column] = meta[column].to_numpy()

            temporary_path = part_path.with_name(
                f"{part_path.stem}.tmp{part_path.suffix}"
            )
            adata_part.write_h5ad(temporary_path, compression="gzip")
            temporary_path.replace(part_path)

            print(
                f"Downloaded {min(start + len(coords), len(expected_ids)):,} "
                f"of {len(expected_ids):,} cells",
                flush=True,
            )

    missing_parts = [path.name for path in part_paths if not path.exists()]
    if missing_parts:
        raise RuntimeError(f"Missing downloaded chunks: {missing_parts[:10]}")


Using existing full-gene raw extract: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_raw.h5ad


In [5]:
if not CENSUS_RAW_H5AD.exists():
    import h5py
    from anndata._io.specs import write_elem

    input_parts = {
        f"part_{i:05d}": str(path)
        for i, path in enumerate(part_paths)
    }

    ad.experimental.concat_on_disk(
        input_parts,
        str(CENSUS_RAW_H5AD),
        axis=0,
        join="inner",
        merge=None,
    )

    # concat_on_disk may omit variable metadata when merge=None.
    first_part = ad.read_h5ad(part_paths[0], backed="r")
    try:
        var_metadata = first_part.var.copy()
        part_var_names = pd.Index(first_part.var_names.astype(str))
    finally:
        first_part.file.close()

    merged = ad.read_h5ad(CENSUS_RAW_H5AD, backed="r")
    try:
        merged_var_names = pd.Index(merged.var_names.astype(str))
    finally:
        merged.file.close()

    if not merged_var_names.equals(part_var_names):
        raise ValueError("Merged genes do not match the downloaded chunk.")

    with h5py.File(CENSUS_RAW_H5AD, "r+") as handle:
        if "var" in handle:
            del handle["var"]
        write_elem(handle, "var", var_metadata)

    print(f"Saved full-gene raw extract: {CENSUS_RAW_H5AD}")
else:
    print("Skipping merge because the full-gene raw extract already exists.")

full_raw = ad.read_h5ad(CENSUS_RAW_H5AD, backed="r")
try:
    if CELL_KEY not in full_raw.obs.columns:
        raise KeyError(f"Raw extract does not contain obs[{CELL_KEY!r}].")
    if "feature_name" not in full_raw.var.columns:
        raise KeyError("Raw extract does not contain var['feature_name'].")

    extracted_ids = pd.to_numeric(
        full_raw.obs[CELL_KEY], errors="raise"
    ).to_numpy(dtype=np.int64)

    if len(extracted_ids) != len(expected_ids):
        raise AssertionError(
            f"Found {len(extracted_ids):,} cells; expected {len(expected_ids):,}."
        )
    if len(np.unique(extracted_ids)) != len(extracted_ids):
        raise AssertionError("The raw extract contains duplicate soma_joinid values.")
    if not np.array_equal(np.sort(extracted_ids), expected_ids):
        missing = np.setdiff1d(expected_ids, extracted_ids)
        extra = np.setdiff1d(extracted_ids, expected_ids)
        raise AssertionError(
            f"Raw cell IDs do not match the sample file. "
            f"Missing: {missing[:10].tolist()}; extra: {extra[:10].tolist()}"
        )

    print(full_raw)
    print(f"All {len(extracted_ids):,} selected cell IDs match.")
finally:
    full_raw.file.close()


Skipping merge because the full-gene raw extract already exists.
AnnData object with n_obs × n_vars = 54225 × 61497 backed at '/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_raw.h5ad'
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'age', 'age_group'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
All 54,225 selected cell IDs match.


In [6]:
# Filter the backed full-gene file before loading the selected matrix into memory.
full_raw = ad.read_h5ad(CENSUS_RAW_H5AD, backed="r")
try:
    if "feature_name" not in full_raw.var.columns:
        raise KeyError("Raw extract does not contain var['feature_name'].")

    feature_symbols = pd.Index(full_raw.var["feature_name"].astype(str))
    is_model_gene = feature_symbols.isin(model_genes)
    # Keep one copy where multiple Census features share the same gene symbol.
    is_first_symbol = ~feature_symbols.duplicated()
    keep_positions = np.flatnonzero(is_model_gene & is_first_symbol)
    matched_symbols = pd.Index(feature_symbols[keep_positions], name=None)

    if len(matched_symbols) == 0:
        raise ValueError("No CellTypist model genes matched Census feature_name.")
    if not matched_symbols.is_unique:
        raise ValueError("Matched model gene symbols are not unique.")

    # AnnData slicing reads only selected genes from the backed H5AD.
    model_raw = full_raw[:, keep_positions].to_memory()
finally:
    full_raw.file.close()

model_raw.var_names = pd.Index(matched_symbols, name=None)
model_raw.obs.index = pd.Index(
    model_raw.obs[CELL_KEY].astype(str),
    name="cell_id",
)

extracted_ids = pd.to_numeric(
    model_raw.obs[CELL_KEY], errors="raise"
).to_numpy(dtype=np.int64)
if len(extracted_ids) != len(expected_ids):
    raise AssertionError(
        f"Found {len(extracted_ids):,} cells; expected {len(expected_ids):,}."
    )
if len(np.unique(extracted_ids)) != len(extracted_ids):
    raise AssertionError("Filtered data contains duplicate soma_joinid values.")
if not np.array_equal(np.sort(extracted_ids), expected_ids):
    missing = np.setdiff1d(expected_ids, extracted_ids)
    extra = np.setdiff1d(extracted_ids, expected_ids)
    raise AssertionError(
        f"Filtered cell IDs do not match the sample file. "
        f"Missing: {missing[:10].tolist()}; extra: {extra[:10].tolist()}"
    )

# Give the var index a name that does not conflict with a metadata column.
var_index_name = "gene_symbol_index"
while var_index_name in model_raw.var.columns:
    var_index_name += "_"
model_raw.var.index.name = var_index_name

model_raw.write_h5ad(MODEL_RAW_H5AD, compression="gzip")

gene_audit = pd.DataFrame({"model_gene": model_genes})
gene_audit["present_in_dataset"] = gene_audit["model_gene"].isin(matched_symbols)
gene_audit.to_csv(MODEL_GENE_AUDIT_CSV, index=False)

print(f"Cells retained: {model_raw.n_obs:,}")
print(f"Matched model genes retained: {model_raw.n_vars:,} of {len(model_genes):,}")
print(f"Model genes absent from dataset: {(~gene_audit['present_in_dataset']).sum():,}")
print(f"All {len(extracted_ids):,} selected cell IDs match the sample file.")
print(f"Saved raw model-gene H5AD: {MODEL_RAW_H5AD}")


Cells retained: 54,225
Matched model genes retained: 6,089 of 6,639
Model genes absent from dataset: 550
All 54,225 selected cell IDs match the sample file.
Saved raw model-gene H5AD: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_model_genes_raw.h5ad


In [7]:
model_prepared = ad.read_h5ad(MODEL_RAW_H5AD)

# Preserve raw counts, then normalize the model-gene matrix to 10,000 counts per cell.
model_prepared.layers["counts"] = model_prepared.X.copy()
sc.pp.normalize_total(model_prepared, target_sum=10_000)
sc.pp.log1p(model_prepared)

# Verify cell IDs and the normalized library size before writing.
prepared_ids = pd.to_numeric(
    model_prepared.obs[CELL_KEY], errors="raise"
).to_numpy(dtype=np.int64)
if len(prepared_ids) != len(expected_ids) or not np.array_equal(
    np.sort(prepared_ids), expected_ids
):
    raise AssertionError("Prepared file cell IDs do not match the sample file.")
if len(np.unique(prepared_ids)) != len(prepared_ids):
    raise AssertionError("Prepared file contains duplicate soma_joinid values.")

first_cell = model_prepared.X[0]
first_cell_values = first_cell.data if hasattr(first_cell, "data") else np.asarray(first_cell).ravel()
first_row_total = np.expm1(first_cell_values).sum()
if not np.isclose(first_row_total, 10_000, atol=1):
    raise AssertionError(
        f"Normalization check failed: first cell sums to {first_row_total:.2f}."
    )

model_prepared.write_h5ad(MODEL_PREPARED_H5AD, compression="gzip")

if model_prepared.n_vars != model_raw.n_vars:
    raise AssertionError("Prepared output has a different number of genes.")
if "counts" not in model_prepared.layers:
    raise AssertionError("Prepared output is missing layers['counts'].")

print(model_prepared)
print(f"Raw model-gene H5AD: {MODEL_RAW_H5AD}")
print(f"Normalized model-gene H5AD: {MODEL_PREPARED_H5AD}")
print(f"All {len(prepared_ids):,} selected cell IDs match the sample file.")
print(f"First cell pre-log normalized total: {first_row_total:,.1f}")


AnnData object with n_obs × n_vars = 54225 × 6089
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'age', 'age_group'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
    layers: 'counts'
Raw model-gene H5AD: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_model_genes_raw.h5ad
Normalized model-gene H5AD: /Users/artsvetkova/Studies/_JKU/3rd_semester/A